In [2]:
print("hello")


hello


In [1]:
import pickle
import random
import gc
from pathlib import Path

adjacencies_dir = Path("../adjacencies")


In [3]:
def is_undirected_sampled(adj, n_edges=200, seed=42):
    """
    Prüft an einer Zufallsstichprobe von ca. n_edges Kanten, ob ein als
    dict/defaultdict gespeicherter Graph ungerichtet ist (für jede Kante
    (u, v) existiert auch die Rückkante (v, u)). Geeignet auch für sehr
    große Graphen, da nicht der gesamte Graph in Sets umgewandelt wird.
    """
    rng = random.Random(seed)
    nodes = list(adj.keys())

    tested = 0
    missing = 0
    while tested < n_edges:
        u = rng.choice(nodes)
        neighbors = adj[u]
        if not neighbors:
            continue
        # Nachbarn können als list oder set gespeichert sein; random.choice
        # braucht eine indizierbare Sequenz.
        if not isinstance(neighbors, (list, tuple)):
            neighbors = list(neighbors)
        v = rng.choice(neighbors)
        tested += 1
        if u not in adj.get(v, ()):
            missing += 1

    return missing == 0


# Läuft automatisch über alle .pkl-Dateien im Ordner - neu hinzugefügte
# Dateien werden bei erneutem Ausführen der Zelle automatisch mit erfasst.
for path in sorted(adjacencies_dir.glob("*.pkl")):
    with open(path, "rb") as f:
        adjacency = pickle.load(f)

    directed = "undirected" if is_undirected_sampled(adjacency) else "directed"
    print(f"{path.name}: {len(adjacency)} Einträge, {directed}")

    del adjacency
    gc.collect()


Slashdot0811.pkl: 77316 Einträge, directed


adjacency_list_uni.pkl: 6095713 Einträge, directed


gpt4o_adj_from_dataset.pkl: 2920221 Einträge, directed


gpt4o_io.pkl: 2657109 Einträge, directed


wiki-topcats.pkl: 1791489 Einträge, directed


In [4]:
with open(adjacencies_dir / "gpt4o_io.pkl", "rb") as f:
    gpt4_uni = pickle.load(f)

n_of_yoshi = gpt4_uni["Yoshitomo Nara"]
#print(n_of_yoshi)
print(len(n_of_yoshi))
n_of_nara = gpt4_uni["Nara Yoshitomo"]
#print(n_of_nara)
print(len(n_of_nara))
n_of_julian = gpt4_uni["Julian Opie"]
#print(n_of_julian)
print(len(n_of_julian))


227375
138442
120547


In [3]:
with open(adjacencies_dir / "gpt4o_io.pkl", "rb") as f:
    gpt4_uni = pickle.load(f)

gpt4_uni["The Way to Happiness"]

['L. Ron Hubbard',
 'Bridge Publications',
 'Scientology principles',
 'Scientology literature',
 '21 precepts',
 'Scientology organizations',
 "L. Ron Hubbard's philosophy",
 'Scientology training materials']

In [4]:
for w in ['L. Ron Hubbard',
 'Bridge Publications',
 'Scientology principles',
 'Scientology literature',
 '21 precepts',
 'Scientology organizations',
 "L. Ron Hubbard's philosophy",
 'Scientology training materials']:

    print(gpt4_uni[w])

['Dianetics: The Modern Science of Mental Health', 'Mary Sue Hubbard', 'Scientology movement', 'Aleister Crowley', 'Los Angeles, California', 'Scientology', 'Golden Book Award', 'Catholic Church', 'Battlefield Earth', 'Mission Earth', 'The Way to Happiness', 'The Science of Survival', 'The Fundamentals of Thought', 'The Creation of Human Ability', 'Self Analysis', 'The Problems of Work', 'The Phoenix Lectures', 'The History of Man', 'Scientology 0-8', 'The Book of Basics', 'The Master Course', 'Advanced Procedures and Axioms', 'The Scientology Handbook', 'The Way to Happiness Handbook', 'The Scientology Volunteer Ministers Handbook', 'The Book of E-Meter', 'The E-Meter in Action', 'The Scientology Dictionary', 'The Scientology Technical Dictionary', 'The Scientology Glossary', 'The Scientology Handbook for Preclears', 'The Scientology Handbook for Auditors', 'The Scientology Handbook for Supervisors', 'The Scientology Handbook for Students', 'The Scientology Handbook for Ministers', 'T

In [7]:
w = 'literature'
print(gpt4_uni[w])

gpt4o = gpt4_uni

['The Way to Happiness']


In [10]:
print(gpt4o["literature"])

['The Way to Happiness']


## Wikipedia-Titel als Sprung-Emulation?

Kann eine externe Titelliste den gleichverteilten Sprung von DURW ersetzen?
Gemessen werden **zwei verschiedene** Quoten:

- **Trefferquote** — Anteil der Titel mit Entsprechung im Graphen. Bestimmt die
  *Kosten*: bei 10 % braucht ein Sprung im Mittel 10 Ziehungen (Rejection Sampling).
- **Abdeckung** — Anteil der Graph-Knoten, die die Liste erreicht. Bestimmt die
  *Gültigkeit*: der Sprung ist gleichverteilt auf dieser Teilmenge, nicht auf V.
  Nur bei Abdeckung nahe 100 % gilt DURWs `π(v) ~ (w + deg_Gu(v))`.

Die Trefferquote lässt sich durch mehr Ziehungen erkaufen, die Abdeckung nicht.

Die Normalisierung ist stufenweise zuschaltbar (`title_overlap.VARIANTS`), damit
der Beitrag jeder Regel einzeln sichtbar wird. Die Spalte *Titel-Koll.* zählt,
wie viele verschiedene Rohtitel eine Stufe zusammengeworfen hat — Treffer, die
nur durch Einebnen entstehen, sind keine.

In [ ]:
import importlib
import title_overlap
importlib.reload(title_overlap)           # damit Änderungen am Modul greifen
from title_overlap import (VARIANTS, Normalizer, compare, format_report,
                           graph_keys, iter_titles, load_titles, overlap)
from graphs import loader

TITLES = title_overlap.TITLES_DIR / "dewiki-20260801-all-titles-in-ns0.txt"
print(TITLES, "->", TITLES.exists())

# Formatprobe: die ersten Titel und ein paar Knotennamen nebeneinander
probe = [t for _, t in zip(range(5), iter_titles(TITLES))]
print("\nTitel :", probe)

### gpt4o_io

Jeder Graph wird einzeln geladen — `loader` hält ohnehin nur einen im RAM.

In [ ]:
g = loader.load_graph("gpt4o_io")
print("Knoten:", f"{g.n_nodes:,}", "| Beispiele:", g.names[:3])

df_4o = compare(g, TITLES)
print()
print(format_report(df_4o, "gpt4o_io"))
df_4o

### gpt4_io

In [ ]:
loader.clear_cache()
g = loader.load_graph("gpt4_io")
print("Knoten:", f"{g.n_nodes:,}", "| Beispiele:", g.names[:3])

df_4 = compare(g, TITLES)
print()
print(format_report(df_4, "gpt4_io"))
df_4

### Vergleich und Bewertung

`nodes_covered` ist die Zahl, an der die Idee hängt: sie ist die Obergrenze
dessen, was ein so emulierter Sprung überhaupt erreichen kann.

In [ ]:
import pandas as pd

both = pd.concat({"gpt4o_io": df_4o, "gpt4_io": df_4}, names=["graph"])
view = both[["titles_matched", "nodes_covered", "n_common", "draws_per_jump"]].copy()
view["titles_matched"] = (view.titles_matched * 100).round(2)
view["nodes_covered"] = (view.nodes_covered * 100).round(2)
view.columns = ["Treffer %", "Abdeckung %", "gemeinsam", "Ziehungen/Sprung"]
display(view)

best = both.xs("+whitespace", level="variante")
for graph, r in best.iterrows():
    verdict = ("brauchbar" if r.nodes_covered > 0.9 else
               "als gleichverteilter Sprung NICHT brauchbar")
    print(f"{graph:10s} Abdeckung {r.nodes_covered:6.2%} -> {verdict}"
          f"   (ein Sprung kostet ~{r.draws_per_jump:.0f} Ziehungen)")

### Gegenprobe: welche Knoten bleiben unerreichbar?

Wenn die Abdeckung niedrig ist, entscheidet die Frage, *welche* Knoten fehlen.
Fehlen sie zufällig, ist der Sprung nur teurer; hängt das Fehlen mit dem Grad
zusammen, ist er zusätzlich verzerrt — und dann trägt DURWs Gewichtung nicht mehr.

In [ ]:
import numpy as np

# "+whitespace" statt "+ohne_klammern": letzteres bringt nur ~0,3 bis 0,8 %-Punkte
# mehr Abdeckung, wirft dafür aber 529k Titel zusammen -- Treffer durch Einebnen.
norm = VARIANTS["+whitespace"]
titles = load_titles(TITLES, norm)

deg = np.diff(g.indptr)                      # g ist noch gpt4_io
hit = np.fromiter((norm.apply(k) in titles for k in g.names), dtype=bool,
                  count=g.n_nodes)

print(f"getroffen: {hit.sum():,} von {g.n_nodes:,} ({hit.mean():.2%})")
print(f"  mittlerer Out-Grad  getroffen: {deg[hit].mean():7.2f}")
print(f"  mittlerer Out-Grad  verfehlt : {deg[~hit].mean():7.2f}")
print(f"  Median              getroffen: {np.median(deg[hit]):7.0f}")
print(f"  Median              verfehlt : {np.median(deg[~hit]):7.0f}")
print(f"\nSackgassen unter den getroffenen: {(deg[hit] == 0).mean():.1%}")
print(f"Sackgassen unter den verfehlten : {(deg[~hit] == 0).mean():.1%}")

del titles

## dewiki gegen enwiki

Die Graphen tragen englische Entitätsnamen, der erste Dump war deutsch — die
niedrige Abdeckung könnte schlicht die falsche Sprache sein. Derselbe Test mit
`enwiki-latest-all-titles-in-ns0.txt` (19,2 Mio Titel statt 5,1 Mio) beantwortet
das. Gleiches Format: Kopfzeile `page_title`, Unterstriche, bereits NFC, keine
echten Leerzeichen.

In [ ]:
DUMPS = {
    "dewiki": title_overlap.TITLES_DIR / "dewiki-20260801-all-titles-in-ns0.txt",
    "enwiki": title_overlap.TITLES_DIR / "enwiki-latest-all-titles-in-ns0.txt",
}
for k, v in DUMPS.items():
    print(f"{k}: {v.exists()}  {v.stat().st_size/2**20:,.0f} MiB")

# Achtung: enwiki hat 19,2 Mio Titel. Eine Variante braucht ~1 Minute und
# einige GB -- die sechs Stufen je Graph laufen entsprechend lang.
results = {}
for dump, path in DUMPS.items():
    for graph in ("gpt4o_io", "gpt4_io"):
        loader.clear_cache()
        g = loader.load_graph(graph)
        results[(dump, graph)] = compare(g, path, log=None)
        print(format_report(results[(dump, graph)], f"{graph} / {dump}"))
        print()

In [ ]:
# Nur die empfohlene Stufe, beide Dumps nebeneinander
rows = []
for (dump, graph), df in results.items():
    r = df.loc["+whitespace"]
    rows.append({"dump": dump, "graph": graph,
                 "Treffer %": round(r.titles_matched * 100, 2),
                 "Abdeckung %": round(r.nodes_covered * 100, 2),
                 "gemeinsam": int(r.n_common),
                 "Ziehungen/Sprung": round(r.draws_per_jump, 1)})
pd.DataFrame(rows).set_index(["graph", "dump"]).sort_index()

### Gradverzerrung der erreichbaren Teilmenge

Entscheidend ist nicht nur *wie viele* Knoten die Liste erreicht, sondern
*welche*. Fehlen bevorzugt Knoten mit kleinem Grad, trifft der emulierte Sprung
gerade den Teil des Graphen nicht, für den er da ist.

In [ ]:
norm = VARIANTS["+whitespace"]
for dump, path in DUMPS.items():
    titles = load_titles(path, norm)
    for graph in ("gpt4o_io", "gpt4_io"):
        loader.clear_cache()
        g = loader.load_graph(graph)
        deg = np.diff(g.indptr)
        hit = np.fromiter((norm.apply(k) in titles for k in g.names),
                          dtype=bool, count=g.n_nodes)
        print(f"{graph} / {dump}: getroffen {hit.mean():6.2%} | "
              f"Out-Grad getroffen {deg[hit].mean():6.2f} vs verfehlt {deg[~hit].mean():6.2f} | "
              f"Sackgassen {(deg[hit]==0).mean():5.1%} vs {(deg[~hit]==0).mean():5.1%}")
    del titles

## Knoten nach In-Degree

Für `gpt4_io` und `gpt4o_io` **alle** Knoten nach Eingangsgrad geordnet als
Textliste unter `additionals/` ablegen, eine Zeile `<eingangsgrad>\t<name>`.
Bausteine: `loader.load_graph` (löst Kürzel auf), `np.bincount(g.indices, …)`
für den Eingangsgrad je Knoten, `np.argsort` für die Rangfolge, `g.names` für
den Namen. Der Ausgangsgrad wäre `np.diff(g.indptr)` — hier nicht gebraucht.
`top=N` kappt auf die N größten.

In [ ]:
import numpy as np
from pathlib import Path
from graphs import loader

OUT_DIR = Path("../additionals")     # neben Code/, wie adjacencies_dir oben


def write_in_degree(graph_name, top=None, names_only=False):
    """Alle Knoten nach Eingangsgrad geordnet -> additionals/<name>__in-degree.txt
    (Zeile '<eingangsgrad>\\t<name>'). top=N kappt auf die N größten."""
    loader.clear_cache()                                    # nur ein Graph im RAM
    g = loader.load_graph(graph_name)                       # Kürzel -> kanonischer Name
    in_deg = np.bincount(g.indices, minlength=g.n_nodes)    # Eingangsgrad je Knoten-ID
    order = np.argsort(in_deg, kind="stable")[::-1]         # absteigend, Ties nach ID
    if top is not None:
        order = order[:top]

    OUT_DIR.mkdir(exist_ok=True)
    path = OUT_DIR / f"{g.name}__in-degree.txt"
    names = g.names
    with path.open("w", encoding="utf-8") as f:
        if names_only:
            f.writelines(f"{names[i]}\n" for i in order)
        else:
            f.writelines(f"{int(in_deg[i])}\t{names[i]}\n" for i in order)

    print(f"{g.name}: |V|={g.n_nodes:,}  {len(order):,} Zeilen  ->  {path}")
    for i in order[:10]:
        print(f"  {int(in_deg[i]):>9,}  {names[i]}")
    return path


for name in ("gpt4_io", "gpt4o_io"):
    write_in_degree(name)          # vollständig; write_in_degree(name, top=1000) kappt
    print()